# 🎾 サーブ3D解析 (GVHMR) — 検証済み最終版

自分のサーブ動画から **world-grounded な3D人体** を復元し、
**物理的に正しい全身重心(COM)** を算出して、沈み込み〜打点を定量化する。

**2026-07-19 に実機で完走を確認済みのレシピ。**

---
### ⚠️ 最重要: Python 3.10 環境が必須
Colab標準の Python 3.12 では **絶対に動きません**。理由:
- `numpy==1.23.5` に py3.12 用 wheel が無くビルドも失敗
- `chumpy` が py3.11+ で削除された `inspect.getargspec` を使用
- `pytorch3d` が cp310 専用 wheel

→ **`uv` で Python 3.10 の仮想環境を作り、そこで GVHMR を動かす。**
ノートブックのカーネルは 3.12 のままでよい（解析は `.npy` 経由で受け渡す）。

---
### 事前準備
1. `ランタイム → ランタイムのタイプを変更 → T4 GPU`
2. body models を各サイトで登録DL（無料）し、**Googleドライブに置く**
   - SMPL: https://smpl.is.tue.mpg.de/ → `SMPL_python_v.1.1.0.zip`
     → 中の `basicmodel_neutral_lbs_10_207_0_v1.1.0.pkl` を **`SMPL_NEUTRAL.pkl`** にリネーム
   - SMPL-X: https://smpl-x.is.tue.mpg.de/ → `models_smplx_v1_1.zip`
     → 中の **`SMPLX_NEUTRAL.npz`**
   - この2ファイルを ドライブの `MyDrive/gvhmr_body_models/` に置く

## 1. GPU 確認

In [ ]:
!nvidia-smi -L
import torch
print('notebook python torch:', torch.__version__, '| cuda:', torch.cuda.is_available())

## 2. GVHMR を取得し、既知のバグを修正

`hmr4d/utils/body_model/body_model.py` の冒頭に `from turtle import forward` という
**IDEの自動インポート誤爆によるゴミ行**がある。`turtle` は tkinter を要求するため
Colab で `ModuleNotFoundError` になる。完全に未使用なので削除する。

In [ ]:
%cd /content
![ -d GVHMR] || git clone https://github.com/zju3dv/GVHMR
%cd /content/GVHMR

# 既知バグ: 不要な turtle import を全 .py から削除
!find /content/GVHMR -name "*.py" -exec sed -i '/from turtle import/d' {} +
!grep -rn "from turtle" /content/GVHMR --include=*.py || echo '✅ turtle import 除去済み'

## 3. ★ Python 3.10 環境を構築（このノートの心臓部）

`chumpy` は setup.py で `pip` を import するのに宣言していないため、
uv のビルド分離下で失敗する。→ 環境に pip を入れて `--no-build-isolation` で個別導入。

所要 5〜10分。

In [ ]:
%cd /content/GVHMR
!pip install -q uv

# Python 3.10 の venv を作成（uv が 3.10 本体を自動取得）
!uv venv --python 3.10 /content/gvhmr-env

# chumpy 対策: 先に pip/setuptools を入れ、ビルド分離を切って導入
!uv pip install --python /content/gvhmr-env/bin/python pip setuptools wheel
!uv pip install --python /content/gvhmr-env/bin/python --no-build-isolation chumpy==0.70

# 残りの依存 + GVHMR 本体（requirements.txt は無改変でOK。3.10 なら全部通る）
!uv pip install --python /content/gvhmr-env/bin/python -r requirements.txt
!uv pip install --python /content/gvhmr-env/bin/python -e .

## 4. 環境の検証

⚠️ スクリプトは **`/content/gvscripts/`** に置く。
`/tmp` に `inspect.py` のような標準ライブラリと同名のファイルがあると、
`import torch` が内部で `import inspect` した際にそちらを掴んで循環インポートで壊れる。

In [ ]:
!mkdir -p /content/gvscripts

In [ ]:
%%writefile /content/gvscripts/gv_check.py
import torch, numpy, pytorch3d, smplx, chumpy, pytorch_lightning, colorlog
print('torch  ', torch.__version__, '| cuda:', torch.cuda.is_available())
print('numpy  ', numpy.__version__)
print('pytorch3d / smplx / chumpy / lightning / colorlog: OK')

In [ ]:
!/content/gvhmr-env/bin/python /content/gvscripts/gv_check.py

**期待する出力** — ここが違ったら先に進まないこと:
```
torch   2.3.0+cu121 | cuda: True
numpy   1.23.5
```

## 5. チェックポイントの取得（HuggingFace）

公式は Google Drive 配布だが **ダウンロード制限に頻繁に引っかかる**。
HF ミラー `ryanrudes/gvhmr` が **同一のディレクトリ構造** で公開しているのでこちらを使う。
合計 約5GB、数分。

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download

for rel in ['gvhmr/gvhmr_siga24_release.ckpt',
            'hmr2/epoch=10-step=25000.ckpt',
            'vitpose/vitpose-h-multi-coco.pth',
            'yolo/yolov8x.pt']:
    p = hf_hub_download(repo_id='ryanrudes/gvhmr', filename=rel,
                        local_dir='/content/GVHMR/inputs/checkpoints')
    print('✅', p)

## 6. Body models を配置

`files.upload()` は 340MB だと極端に遅く失敗もするため、
**Googleドライブ経由**（Google内部コピーなので数秒）を使う。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
SRC = '/content/drive/MyDrive/gvhmr_body_models'   # ← 事前にここへ2ファイルを置く
DST = '/content/GVHMR/inputs/checkpoints/body_models'
os.makedirs(f'{DST}/smplx', exist_ok=True)
os.makedirs(f'{DST}/smpl',  exist_ok=True)

shutil.copy(f'{SRC}/SMPLX_NEUTRAL.npz', f'{DST}/smplx/')
shutil.copy(f'{SRC}/SMPL_NEUTRAL.pkl',  f'{DST}/smpl/')
!find {DST} -type f -exec ls -lh {{}} \;

## 7. 全ファイルの確認

In [ ]:
!find /content/GVHMR/inputs/checkpoints -type f -exec ls -lh {} \;

**揃っているべきもの:**

| パス | 用途 |
|---|---|
| `gvhmr/gvhmr_siga24_release.ckpt` | GVHMR本体 |
| `hmr2/epoch=10-step=25000.ckpt` | 特徴抽出 |
| `vitpose/vitpose-h-multi-coco.pth` | 2D姿勢 |
| `yolo/yolov8x.pt` | 人物検出 |
| `body_models/smplx/SMPLX_NEUTRAL.npz` | **推論の本体**（体を生成） |
| `body_models/smpl/SMPL_NEUTRAL.pkl` | 描画メッシュ＋関節の取り出し先 |

`dpvo/` は `-s`（静止カメラ）を使うため不要。

## 8. 💾 バックアップ（強く推奨）

セッションが切れると5GB落とし直しになる。ドライブへ退避しておけば数十秒で復旧できる。

In [ ]:
import shutil
shutil.copytree('/content/GVHMR/inputs/checkpoints',
                '/content/drive/MyDrive/gvhmr_backup/checkpoints',
                dirs_exist_ok=True)
print('✅ バックアップ完了')
!du -sh /content/drive/MyDrive/gvhmr_backup/checkpoints

## 9. スモークテスト（同梱の tennis.mp4）

`-s` = 静止カメラ。SLAM(DPVO) を回避でき、三脚固定の撮影ならこれで十分。
**必ず venv の python を明示**して実行する。

In [ ]:
%cd /content/GVHMR
!/content/gvhmr-env/bin/python tools/demo/demo.py --video=docs/example_video/tennis.mp4 -s 2>&1 | tail -25

`Rendering Incam` と `Rendering Global` が 100% まで進めば環境構築は完了。

## 10. 自分のサーブ動画を解析

三脚固定・全身が映る・1人が明瞭に写っている動画が望ましい
（背景に別の人がいると人物追跡が移る場合がある）。

In [ ]:
import os
os.makedirs('/content/GVHMR/inputs/serve', exist_ok=True)
from google.colab import files
up = files.upload()
for name in up:
    os.replace(name, f'/content/GVHMR/inputs/serve/{name}')
    print('✅', name)

In [ ]:
%cd /content/GVHMR
import glob, os
serve = sorted(glob.glob('inputs/serve/*'), key=os.path.getmtime)[-1]
NAME  = os.path.splitext(os.path.basename(serve))[0]
OUTDIR = f'/content/GVHMR/outputs/demo/{NAME}'
print('対象:', serve, '\n出力:', OUTDIR)
!/content/gvhmr-env/bin/python tools/demo/demo.py --video="{serve}" -s 2>&1 | tail -25

## 11. 3D重心(COM)の算出

GVHMR の出力 `smpl_params_global` は **名前に反して SMPL-X パラメータ**
（`body_pose` が 63 = 21関節×3。SMPL なら 69）。
そのため GVHMR 本体と同じ経路をたどる:

```
SMPL-X で推論 → smplx2smpl で SMPL頂点へ変換 → J_regressor で24関節
→ De Leva の体節質量比で全身重心
```

体節質量比（合計 1.000）: 体幹 49.7% / 頭 8.1% / 大腿 各10% / 下腿 各4.65% / 上腕 各2.8% ほか。

In [ ]:
%%writefile /content/gvscripts/gv_com.py
import os, sys, torch, numpy as np
os.chdir('/content/GVHMR'); sys.path.insert(0, '/content/GVHMR')
from hmr4d.utils.smplx_utils import make_smplx

OUT = sys.argv[1]
res = torch.load(f'{OUT}/hmr4d_results.pt', map_location='cpu')
g   = res['smpl_params_global']
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

# SMPL-X で体を生成 → SMPL頂点 → 24関節
model = make_smplx('supermotion').to(dev)
with torch.no_grad():
    out = model(**{k: v.to(dev) for k, v in g.items()})
s2s    = torch.load('hmr4d/utils/body_model/smplx2smpl_sparse.pt').to(dev)
verts  = torch.stack([torch.matmul(s2s, v) for v in out.vertices])
Jreg   = torch.load('hmr4d/utils/body_model/smpl_neutral_J_regressor.pt').to(dev)
joints = torch.einsum('jv,fvc->fjc', Jreg, verts).cpu().numpy()   # (F,24,3) meters
print('joints:', joints.shape)

# 上方向の軸を実測（頭 - 足首）
up_vec = (joints[:, 15] - (joints[:, 7] + joints[:, 8]) / 2).mean(0)
up_ax  = int(np.argmax(np.abs(up_vec))); up_sgn = float(np.sign(up_vec[up_ax]))
print(f'up vector {up_vec.round(3)} -> axis {up_ax} sign {up_sgn:+.0f}')

# De Leva 体節質量比 (親関節, 子関節, 質量比, 近位からのCOM比)
SEG = [(0,12,0.497,0.50), (12,15,0.081,0.50),
       (16,18,0.028,0.436), (17,19,0.028,0.436),
       (18,20,0.016,0.430), (19,21,0.016,0.430),
       (20,22,0.006,0.50),  (21,23,0.006,0.50),
       (1,4,0.100,0.433),   (2,5,0.100,0.433),
       (4,7,0.0465,0.433),  (5,8,0.0465,0.433),
       (7,10,0.0145,0.50),  (8,11,0.0145,0.50)]
com = np.zeros((joints.shape[0], 3))
for a, b, m, r in SEG:
    com += m * (joints[:, a] * (1 - r) + joints[:, b] * r)
com /= sum(s[2] for s in SEG)

np.save('/content/gv_joints.npy', joints)
np.save('/content/gv_com.npy', com)
np.save('/content/gv_upaxis.npy', np.array([up_ax, up_sgn]))
print('COM mean:', com.mean(0).round(3), 'm')
print('✅ saved gv_joints.npy / gv_com.npy / gv_upaxis.npy')

In [ ]:
!/content/gvhmr-env/bin/python /content/gvscripts/gv_com.py "{OUTDIR}"

## 12. 沈み込みと打点を自動検出

In [ ]:
import numpy as np
com = np.load('/content/gv_com.npy')
up_ax, up_sgn = np.load('/content/gv_upaxis.npy'); up_ax = int(up_ax)
h = com[:, up_ax] * up_sgn
FPS = 30

pk = int(np.argmax(h))                       # 最高点 = 打点
tr = int(np.argmin(h[:pk])) if pk > 0 else 0 # その手前の最低点 = 沈み込み
v  = np.gradient(h) * FPS

print(f'沈み込み : frame {tr:3d}  {h[tr]:.3f} m  ({tr/FPS:.2f}s)')
print(f'打点     : frame {pk:3d}  {h[pk]:.3f} m  ({pk/FPS:.2f}s)')
print(f'伸び上がり: {h[pk]-h[tr]:.3f} m / {(pk-tr)/FPS:.2f} 秒')
print(f'最大上昇速度: {v[tr:pk+1].max():.2f} m/s')
print(f'重心の高さ範囲: {h.min():.3f} - {h.max():.3f} m')

## 13. 検証: 検出フレームが実際の動作と一致するか

**ここがP0の判定**。重心の谷＝沈み込み、山＝打点 になっていれば、
3D重心が物理的に正しく計算できている証拠。

In [ ]:
import cv2, matplotlib.pyplot as plt
cap = cv2.VideoCapture(f'{OUTDIR}/0_input_video.mp4')
targets = [tr, pk-8, pk-4, pk, pk+5]
fig, axes = plt.subplots(1, len(targets), figsize=(20, 7))
for ax, f in zip(axes, targets):
    cap.set(cv2.CAP_PROP_POS_FRAMES, f)
    ok, img = cap.read()
    if ok:
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        tag = '  <- LOADING' if f == tr else ('  <- CONTACT' if f == pk else '')
        ax.set_title(f'frame {f}{tag}')
    ax.axis('off')
cap.release(); plt.tight_layout(); plt.show()

## 14. 注釈つきグラフ（人が読める形）

In [ ]:
import numpy as np, matplotlib.pyplot as plt
t = np.arange(len(h)) / FPS

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(t, h, lw=2.5, color='#2563eb')
ax.axvspan(tr/FPS, pk/FPS, color='orange', alpha=.15)
ax.scatter([tr/FPS], [h[tr]], s=140, color='orange', zorder=5)
ax.scatter([pk/FPS], [h[pk]], s=140, color='red', zorder=5)
ax.annotate(f'LOADING\nknees bent\n{h[tr]:.3f} m',
            xy=(tr/FPS, h[tr]), xytext=(tr/FPS-1.5, h.min()+0.01),
            arrowprops=dict(arrowstyle='->', lw=1.5), fontsize=11)
ax.annotate(f'CONTACT\nfully extended\n{h[pk]:.3f} m',
            xy=(pk/FPS, h[pk]), xytext=(pk/FPS+0.5, h[pk]+0.005),
            arrowprops=dict(arrowstyle='->', lw=1.5), fontsize=11)
ax.text((tr+pk)/2/FPS, h.min()-0.005,
        f'LEG DRIVE  +{(h[pk]-h[tr])*100:.1f}cm in {(pk-tr)/FPS:.2f}s',
        ha='center', fontsize=11, color='darkorange', weight='bold')
ax.set_xlabel('time (s)'); ax.set_ylabel('center of mass height (m)')
ax.set_title('Serve — center of mass height')
ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## 15. 結果のバックアップ

In [ ]:
import shutil, os
DEST = '/content/drive/MyDrive/gvhmr_backup/results'
os.makedirs(DEST, exist_ok=True)
for f in ['/content/gv_joints.npy', '/content/gv_com.npy', '/content/gv_upaxis.npy']:
    shutil.copy(f, DEST)
shutil.copy(f'{OUTDIR}/hmr4d_results.pt', DEST)
print('✅ 保存先:', DEST)
!ls -lh {DEST}

---
## 付録A: 次のセッションでの復旧手順

チェックポイントをドライブに退避してあれば、5GBの再取得は不要:

1. セクション 1〜4 を実行（clone / turtle修正 / Python3.10環境 / 検証）
2. セクション 5・6 の代わりに下のセルを実行
3. セクション 9 以降へ

In [ ]:
# 復旧用（セクション5・6の代替）
from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.copytree('/content/drive/MyDrive/gvhmr_backup/checkpoints',
                '/content/GVHMR/inputs/checkpoints', dirs_exist_ok=True)
!find /content/GVHMR/inputs/checkpoints -type f -exec ls -lh {} \;

---
## 付録B: 詰まったときの既知の対処

| 症状 | 原因 | 対処 |
|---|---|---|
| `No module named 'pytorch_lightning'` 等 | `pip -r requirements.txt` が途中で中断 | py3.10 の venv で入れ直す（セクション3） |
| `not a supported wheel on this platform` | Python 3.12 で実行している | **venv の python を明示**して実行 |
| `No module named 'tkinter'` | `from turtle import forward` のゴミ行 | セクション2 の sed で削除 |
| `partially initialized module 'torch'` | `/tmp/inspect.py` 等が標準ライブラリを隠蔽 | `/content/gvscripts/` に置く。衝突名を避ける |
| `Too many users have viewed...` | Google Drive のDL制限 | HF ミラー `ryanrudes/gvhmr` を使う（セクション5） |
| `cannot import name '_CAFFE2_ATEN_FALLBACK'` | torch が壊れている | `pip install --force-reinstall torch` → セッション再起動 |
| Failed to build `chumpy` | setup.py が pip を未宣言 | `--no-build-isolation` で個別導入（セクション3） |

---
## 付録C: 用語

- **SMPL / SMPL-X**: 3D人体モデル。SMPL-X = SMPL + 手指 + 顔。GVHMR は SMPL-X で推論し SMPL 形式へ変換する
- **world-grounded**: 重力方向を推定した世界座標系。重心・バランス分析に必須
- **COM (center of mass)**: 全身重心。体節ごとの質量比から算出する物理量
- **`-s` フラグ**: 静止カメラ想定で SLAM をスキップ

設計全体は リポジトリの `REDESIGN.md` を参照。